# OTEL Synthetic Data Generator

Generate synthetic OpenTelemetry span data for the air-gap Dask/Panel-Viz stack.

**Methodology**: Same vectorized NumPy generation as the production 1TB+ dataset
(K8s Indexed Job with 8 parallel pods). Adapted for single-node JupyterHub execution.

**Output**: Hive-partitioned Parquet files in S3, compatible with OTEL Navigator (Panel-Viz).

| Parameter | Default | Description |
|-----------|---------|-------------|
| `TOTAL_SPANS` | 500,000 | Total spans to generate (50K quick test, 5M+ stress) |
| `CHUNK_SIZE` | 200,000 | Spans per chunk (controls peak memory) |
| `DURATION_DAYS` | 2 | Time range for generated spans |
| `PREFIX` | `otel-notebook` | S3 prefix (distinct from production `otel-minimal`) |

**Note**: This notebook is read-only from ConfigMap. To edit:
```bash
cp ~/sample-notebooks/OTEL_Data_Generator.ipynb ~/
```

In [ ]:
# --- User Configuration ---
TOTAL_SPANS = 500_000       # 500K default (50K for quick test, 5M+ for stress)
CHUNK_SIZE = 200_000        # Spans per chunk (controls peak memory ~300 MB)
DURATION_DAYS = 2           # Time range for generated spans
COMPRESSION = "ZSTD"
COMPRESSION_LEVEL = 3
ROW_GROUP_SIZE = 100_000

# S3 target (reads from JupyterHub env vars, override as needed)
import os
BUCKET = os.getenv('S3_BUCKET', 'cyberphy')
PREFIX = "otel-notebook"    # Distinct prefix to avoid overwriting production data
S3_ENDPOINT = os.getenv('S3_ENDPOINT', '')
S3_REGION = os.getenv('AWS_REGION', os.getenv('S3_REGION', 'us-east-1'))

print(f"Target: s3://{BUCKET}/{PREFIX}/spans/")
print(f"Spans:  {TOTAL_SPANS:,} in chunks of {CHUNK_SIZE:,}")
print(f"Days:   {DURATION_DAYS}")
print(f"S3:     {S3_ENDPOINT or 'AWS S3 (default)'}")

In [ ]:
import json
import random
import sys
import time
from datetime import datetime, timedelta, timezone

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
import pyarrow.fs as pafs

# Schema (30 columns — identical to production 1TB dataset)
_span_kind_dict = pa.dictionary(pa.int8(), pa.string())
_status_code_dict = pa.dictionary(pa.int8(), pa.string())

SPANS_SCHEMA = pa.schema([
    pa.field("trace_id", pa.string(), nullable=False),
    pa.field("span_id", pa.string(), nullable=False),
    pa.field("parent_span_id", pa.string(), nullable=True),
    pa.field("start_time_unix_nano", pa.int64(), nullable=False),
    pa.field("end_time_unix_nano", pa.int64(), nullable=False),
    pa.field("duration_ns", pa.int64(), nullable=False),
    pa.field("name", pa.string(), nullable=False),
    pa.field("kind", _span_kind_dict, nullable=False),
    pa.field("status_code", _status_code_dict, nullable=False),
    pa.field("status_message", pa.string(), nullable=True),
    pa.field("service_name", pa.string(), nullable=False),
    pa.field("service_namespace", pa.string(), nullable=True),
    pa.field("service_version", pa.string(), nullable=True),
    pa.field("host_name", pa.string(), nullable=True),
    pa.field("host_ip", pa.string(), nullable=True),
    pa.field("attributes_json", pa.string(), nullable=True),
    pa.field("resource_attributes_json", pa.string(), nullable=True),
    pa.field("http_method", pa.string(), nullable=True),
    pa.field("http_status_code", pa.int16(), nullable=True),
    pa.field("http_url", pa.string(), nullable=True),
    pa.field("http_route", pa.string(), nullable=True),
    pa.field("http_target", pa.string(), nullable=True),
    pa.field("db_system", pa.string(), nullable=True),
    pa.field("db_name", pa.string(), nullable=True),
    pa.field("db_operation", pa.string(), nullable=True),
    pa.field("db_statement", pa.string(), nullable=True),
    pa.field("rpc_system", pa.string(), nullable=True),
    pa.field("rpc_service", pa.string(), nullable=True),
    pa.field("rpc_method", pa.string(), nullable=True),
    pa.field("messaging_system", pa.string(), nullable=True),
    pa.field("messaging_destination", pa.string(), nullable=True),
    pa.field("messaging_operation", pa.string(), nullable=True),
    pa.field("exception_type", pa.string(), nullable=True),
    pa.field("exception_message", pa.string(), nullable=True),
    pa.field("events_count", pa.int32(), nullable=True),
    pa.field("links_count", pa.int32(), nullable=True),
])

# 12 service templates (identical to production)
SERVICE_TEMPLATES = [
    {"name": "api-gateway",          "kind": "SERVER",   "operations": ["HTTP request", "route", "rate-limit"]},
    {"name": "auth-service",         "kind": "SERVER",   "operations": ["validate_token", "refresh_token", "login", "logout"]},
    {"name": "user-service",         "kind": "SERVER",   "operations": ["get_user", "update_user", "list_users", "delete_user"]},
    {"name": "order-service",        "kind": "SERVER",   "operations": ["create_order", "get_order", "process_order", "cancel_order"]},
    {"name": "inventory-service",    "kind": "SERVER",   "operations": ["check_stock", "reserve_item", "release_item"]},
    {"name": "payment-service",      "kind": "SERVER",   "operations": ["process_payment", "refund", "validate_card"]},
    {"name": "notification-service", "kind": "PRODUCER", "operations": ["send_email", "send_sms", "send_push"]},
    {"name": "shipping-service",     "kind": "SERVER",   "operations": ["calculate_shipping", "create_shipment", "track"]},
    {"name": "analytics-service",    "kind": "CONSUMER", "operations": ["process_event", "aggregate", "report"]},
    {"name": "database",             "kind": "CLIENT",   "operations": ["SELECT", "INSERT", "UPDATE", "DELETE"]},
    {"name": "cache",                "kind": "CLIENT",   "operations": ["GET", "SET", "DEL", "MGET", "SCAN"]},
    {"name": "message-queue",        "kind": "PRODUCER", "operations": ["publish", "consume", "ack", "nack"]},
]

# Pre-compute flat arrays for vectorized lookups
SVC_NAMES = [s["name"] for s in SERVICE_TEMPLATES]
SVC_KINDS = [s["kind"] for s in SERVICE_TEMPLATES]
SVC_OPS, SVC_OP_IDX = [], []
for si, s in enumerate(SERVICE_TEMPLATES):
    for op in s["operations"]:
        SVC_OPS.append(op)
        SVC_OP_IDX.append(si)
SVC_OPS = np.array(SVC_OPS)
SVC_OP_IDX = np.array(SVC_OP_IDX)

HTTP_METHODS = np.array(["GET", "POST", "PUT", "DELETE", "PATCH"])
HTTP_ROUTES = np.array([
    "/api/users", "/api/users/{id}", "/api/orders", "/api/orders/{id}",
    "/api/products", "/api/products/{id}", "/api/auth/login", "/api/auth/logout",
    "/api/health", "/api/metrics", "/api/cart", "/api/checkout",
])
HTTP_STATUS_CODES = np.array([200, 200, 200, 200, 201, 204, 400, 401, 403, 404, 500, 502, 503], dtype=np.int16)

service_count = len(SERVICE_TEMPLATES)
print(f"Schema: {len(SPANS_SCHEMA)} columns")
print(f"Services: {service_count} templates, {len(SVC_OPS)} total operations")

In [ ]:
def generate_batch_vectorized(n, rng, start_ns, time_range_ns):
    """Generate n spans using vectorized numpy operations.

    Same methodology as the production 1TB K8s Indexed Job:
    - Exponential duration distribution (median ~50ms)
    - 5% error rate
    - Rejection sampling for service-operation matching
    - Full 30-column OTEL schema
    """
    # Timing: uniform start offset, exponential duration
    offsets_ns = rng.integers(0, time_range_ns, size=n, dtype=np.int64)
    span_starts = start_ns + offsets_ns
    durations = np.clip(
        rng.exponential(50_000_000, size=n).astype(np.int64),  # median ~50ms
        1_000_000, 30_000_000_000,  # 1ms to 30s
    )
    span_ends = span_starts + durations

    # Service assignment
    svc_indices = rng.integers(0, service_count, size=n)
    is_error = rng.random(size=n) < 0.05

    # Vectorized rejection sampling for service-operation matching
    op_global_indices = rng.integers(0, len(SVC_OPS), size=n)
    mismatched = SVC_OP_IDX[op_global_indices] != svc_indices
    retries = 0
    while mismatched.any() and retries < 20:
        new_indices = rng.integers(0, len(SVC_OPS), size=mismatched.sum())
        op_global_indices[mismatched] = new_indices
        mismatched = SVC_OP_IDX[op_global_indices] != svc_indices
        retries += 1
    if mismatched.any():
        for idx in np.where(mismatched)[0]:
            si = svc_indices[idx]
            candidates = np.where(SVC_OP_IDX == si)[0]
            op_global_indices[idx] = rng.choice(candidates)

    names = SVC_OPS[op_global_indices]
    kinds_arr = np.array(SVC_KINDS)[svc_indices]
    svc_names_arr = np.array(SVC_NAMES)[svc_indices]

    # Boolean masks for conditional fields
    is_server = kinds_arr == "SERVER"
    is_client = kinds_arr == "CLIENT"
    is_producer = kinds_arr == "PRODUCER"
    is_consumer = kinds_arr == "CONSUMER"
    is_msg = is_producer | is_consumer

    # HTTP fields (SERVER spans only)
    http_method_idx = rng.integers(0, len(HTTP_METHODS), size=n)
    http_route_idx = rng.integers(0, len(HTTP_ROUTES), size=n)
    http_status_idx = rng.integers(0, len(HTTP_STATUS_CODES), size=n)

    # Host fields
    host_nums = rng.integers(1, 11, size=n)
    ip_b = rng.integers(1, 256, size=n)
    ip_c = rng.integers(1, 256, size=n)

    # Trace/span IDs via numpy random bytes
    trace_raw = rng.bytes(n * 16)
    trace_ids = [trace_raw[i*16:(i+1)*16].hex() for i in range(n)]
    span_raw = rng.bytes(n * 8)
    span_ids = [span_raw[i*8:(i+1)*8].hex() for i in range(n)]

    events_count = rng.integers(0, 4, size=n, dtype=np.int32)
    links_count = np.zeros(n, dtype=np.int32)

    # Vectorized column construction
    svc_names_list = svc_names_arr.tolist()
    names_list = names.tolist()
    status_codes = np.where(is_error, "ERROR", "OK").tolist()
    status_msgs = np.where(is_error, "Error occurred", None).tolist()

    host_names_list = np.char.add(
        np.char.add(svc_names_arr.astype(str), "-"),
        np.char.add(host_nums.astype(str), ".example.com")
    ).tolist()
    host_ips = np.char.add(
        "10.0.",
        np.char.add(np.char.add(ip_b.astype(str), "."), ip_c.astype(str))
    ).tolist()

    http_method_vals = HTTP_METHODS[http_method_idx]
    http_route_vals = HTTP_ROUTES[http_route_idx]
    http_status_vals = HTTP_STATUS_CODES[http_status_idx]

    http_methods = np.where(is_server, http_method_vals, None).tolist()
    http_routes_list = np.where(is_server, http_route_vals, None).tolist()
    http_statuses = np.where(is_server, http_status_vals, None).tolist()
    http_urls = np.where(is_server, np.char.add("https://api.example.com", http_route_vals), None).tolist()

    attrs = np.char.add(np.char.add('{"service.name": "', svc_names_arr.astype(str)), '"}').tolist()
    res_attrs = ['{}'] * n

    db_systems = np.where(is_client, "postgresql", None).tolist()
    db_names_list = np.where(is_client, "production", None).tolist()
    db_ops = np.where(is_client, names, None).tolist()

    msg_systems = np.where(is_msg, "kafka", None).tolist()
    msg_dests = np.where(is_msg, "events", None).tolist()
    msg_ops = np.where(is_producer, "publish", np.where(is_consumer, "consume", None)).tolist()

    exc_types = np.where(is_error, "RuntimeError", None).tolist()
    exc_msgs = np.where(is_error, "Something went wrong", None).tolist()

    # Compute date partition column
    timestamps_s = span_starts // 1_000_000_000
    days_since_epoch = (timestamps_s // 86400).astype(np.int32)
    hours_of_day = ((timestamps_s % 86400) // 3600).astype(np.int8)
    unique_days = np.unique(days_since_epoch)
    day_to_str = {}
    for d in unique_days:
        dt = datetime.fromtimestamp(int(d) * 86400, tz=timezone.utc)
        day_to_str[int(d)] = dt.strftime("%Y-%m-%d")
    date_col = pa.array([day_to_str[int(d)] for d in days_since_epoch], type=pa.string())
    hour_col = pa.array(hours_of_day, type=pa.int8())

    table = pa.table({
        "trace_id": pa.array(trace_ids, type=pa.string()),
        "span_id": pa.array(span_ids, type=pa.string()),
        "parent_span_id": pa.nulls(n, type=pa.string()),
        "start_time_unix_nano": pa.array(span_starts),
        "end_time_unix_nano": pa.array(span_ends),
        "duration_ns": pa.array(durations),
        "name": pa.array(names_list, type=pa.string()),
        "kind": pa.array(kinds_arr.tolist()).dictionary_encode().cast(_span_kind_dict),
        "status_code": pa.array(status_codes).dictionary_encode().cast(_status_code_dict),
        "status_message": pa.array(status_msgs, type=pa.string()),
        "service_name": pa.array(svc_names_list, type=pa.string()),
        "service_namespace": pa.array(["production"] * n, type=pa.string()),
        "service_version": pa.array(["1.0.0"] * n, type=pa.string()),
        "host_name": pa.array(host_names_list, type=pa.string()),
        "host_ip": pa.array(host_ips, type=pa.string()),
        "attributes_json": pa.array(attrs, type=pa.string()),
        "resource_attributes_json": pa.array(res_attrs, type=pa.string()),
        "http_method": pa.array(http_methods, type=pa.string()),
        "http_status_code": pa.array(http_statuses, type=pa.int16()),
        "http_url": pa.array(http_urls, type=pa.string()),
        "http_route": pa.array(http_routes_list, type=pa.string()),
        "http_target": pa.nulls(n, type=pa.string()),
        "db_system": pa.array(db_systems, type=pa.string()),
        "db_name": pa.array(db_names_list, type=pa.string()),
        "db_operation": pa.array(db_ops, type=pa.string()),
        "db_statement": pa.nulls(n, type=pa.string()),
        "rpc_system": pa.nulls(n, type=pa.string()),
        "rpc_service": pa.nulls(n, type=pa.string()),
        "rpc_method": pa.nulls(n, type=pa.string()),
        "messaging_system": pa.array(msg_systems, type=pa.string()),
        "messaging_destination": pa.array(msg_dests, type=pa.string()),
        "messaging_operation": pa.array(msg_ops, type=pa.string()),
        "exception_type": pa.array(exc_types, type=pa.string()),
        "exception_message": pa.array(exc_msgs, type=pa.string()),
        "events_count": pa.array(events_count),
        "links_count": pa.array(links_count),
        "date": date_col,
        "hour": hour_col,
    })

    return table

print("generate_batch_vectorized() defined")
print("  - Exponential duration (median ~50ms, clipped 1ms-30s)")
print("  - 5% error rate")
print("  - Rejection sampling for service-operation matching")

In [ ]:
# Connect to S3
s3_kwargs = dict(
    access_key=os.environ.get('AWS_ACCESS_KEY_ID', ''),
    secret_key=os.environ.get('AWS_SECRET_ACCESS_KEY', ''),
    region=S3_REGION,
)
session_token = os.environ.get('AWS_SESSION_TOKEN', '')
if session_token:
    s3_kwargs['session_token'] = session_token
if S3_ENDPOINT:
    s3_kwargs['endpoint_override'] = S3_ENDPOINT
    s3_kwargs['scheme'] = 'https' if 'https' in S3_ENDPOINT else 'http'

s3 = pafs.S3FileSystem(**s3_kwargs)
base_path = f"{BUCKET}/{PREFIX}/spans"
print(f"Connected to S3")
print(f"Base path: {base_path}")

def write_partitioned(table, root_path, batch_id, chunk_idx=None, max_retries=5):
    """Write date-partitioned parquet — sorted, ZSTD compressed.

    Layout: root_path/date=YYYY-MM-DD/batch_NNNN_chunk_CC.parquet

    Sorting by start_time_unix_nano enables predicate pushdown via
    row-group statistics (hour-level filtering without Hive partition).
    """
    sort_indices = pc.sort_indices(table, sort_keys=[("start_time_unix_nano", "ascending")])
    table = table.take(sort_indices)

    date_col = table.column("date")
    data_cols = [c for c in table.column_names if c not in ("date", "hour")]
    unique_dates = pc.unique(date_col).to_pylist()

    for date_val in unique_dates:
        date_mask = pc.equal(date_col, date_val)
        partition = table.filter(date_mask).select(data_cols)

        sort_idx = pc.sort_indices(partition, sort_keys=[("start_time_unix_nano", "ascending")])
        partition = partition.take(sort_idx)

        if chunk_idx is not None:
            part_path = f"{root_path}/date={date_val}/batch_{batch_id:04d}_chunk_{chunk_idx:02d}.parquet"
        else:
            part_path = f"{root_path}/date={date_val}/batch_{batch_id:04d}.parquet"

        for attempt in range(max_retries):
            try:
                pq.write_table(
                    partition, part_path, filesystem=s3,
                    row_group_size=ROW_GROUP_SIZE,
                    compression=COMPRESSION,
                    compression_level=COMPRESSION_LEVEL,
                    write_statistics=True,
                    write_page_index=True,
                )
                break
            except OSError as e:
                if "SLOW_DOWN" in str(e) and attempt < max_retries - 1:
                    wait = (2 ** attempt) + random.uniform(0, 2)
                    print(f"  S3 SLOW_DOWN, retry {attempt+1}/{max_retries} after {wait:.1f}s")
                    time.sleep(wait)
                else:
                    raise
        del partition

print("write_partitioned() defined")

In [ ]:
# Time range
end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(days=DURATION_DAYS)
time_range_ns = DURATION_DAYS * 24 * 3600 * 1_000_000_000
start_ns = int(start_time.timestamp() * 1_000_000_000)

num_batches = (TOTAL_SPANS + CHUNK_SIZE - 1) // CHUNK_SIZE

print("=" * 60)
print("OTEL Data Generator")
print("=" * 60)
print(f"Total spans:  {TOTAL_SPANS:,}")
print(f"Chunks:       {num_batches} x {CHUNK_SIZE:,}")
print(f"Time range:   {start_time:%Y-%m-%d %H:%M} to {end_time:%Y-%m-%d %H:%M}")
print(f"Compression:  {COMPRESSION} (level {COMPRESSION_LEVEL})")
print(f"Destination:  s3://{base_path}/")
print("=" * 60)
sys.stdout.flush()

overall_start = time.time()
total_bytes = 0
total_spans_written = 0
remaining = TOTAL_SPANS

for batch_id in range(num_batches):
    chunk_size = min(CHUNK_SIZE, remaining)
    remaining -= chunk_size

    rng = np.random.default_rng(seed=batch_id * 10000)
    t0 = time.time()

    table = generate_batch_vectorized(chunk_size, rng, start_ns, time_range_ns)
    gen_time = time.time() - t0

    t1 = time.time()
    write_partitioned(table, base_path, batch_id, chunk_idx=0)
    write_time = time.time() - t1

    batch_bytes = table.nbytes // 4  # rough ZSTD ratio
    total_bytes += batch_bytes
    total_spans_written += chunk_size
    del table

    pct = total_spans_written / TOTAL_SPANS * 100
    rate = chunk_size / (gen_time + write_time)
    print(f"  [{batch_id+1}/{num_batches}] {chunk_size:,} spans | "
          f"gen {gen_time:.1f}s + write {write_time:.1f}s | "
          f"{rate:,.0f} spans/s | {pct:.0f}%")
    sys.stdout.flush()

# Write _active_dataset.json marker
marker = {
    "dataset": PREFIX,
    "updated_at": datetime.now(timezone.utc).isoformat(),
    "phase": "notebook",
    "total_spans": total_spans_written,
    "total_bytes": total_bytes,
}
marker_key = f"{BUCKET}/_active_dataset.json"
with s3.open_output_stream(marker_key) as f:
    f.write(json.dumps(marker).encode())

elapsed = time.time() - overall_start
print("=" * 60)
print("Generation Complete!")
print("=" * 60)
print(f"Spans:    {total_spans_written:,}")
print(f"Size:     ~{total_bytes / (1024**2):.0f} MiB (compressed)")
print(f"Time:     {elapsed:.1f}s")
print(f"Rate:     {total_spans_written / elapsed:,.0f} spans/s")
print(f"Marker:   s3://{marker_key}")
print(f"Panel-Viz will auto-detect this dataset on next poll (60s)")

## Verify Generated Data

Connect to the Dask cluster and verify the data reads correctly.
The heatmap below uses the same format as OTEL Navigator (Panel-Viz):
`timestamp_s` vs `duration_ms` with datashader.

In [ ]:
import s3fs
import dask.dataframe as dd
from dask.distributed import Client
import holoviews as hv
from holoviews.operation.datashader import datashade, dynspread
hv.extension('bokeh')

# Connect to Dask cluster
scheduler_addr = os.getenv('DASK_SCHEDULER_ADDRESS',
                           'tcp://cybersec-dask-scheduler.dask.svc.cluster.local:8786')
print(f"Connecting to Dask: {scheduler_addr}")
client = Client(scheduler_addr)
print(f"Dashboard: {client.dashboard_link}")
print(f"Workers: {len(client.scheduler_info()['workers'])}")

# Read generated data
s3fs_kwargs = {
    'anon': False,
    'key': os.environ.get('AWS_ACCESS_KEY_ID') or None,
    'secret': os.environ.get('AWS_SECRET_ACCESS_KEY') or None,
    'token': os.environ.get('AWS_SESSION_TOKEN') or None,
}
if S3_ENDPOINT:
    s3fs_kwargs['client_kwargs'] = {'endpoint_url': S3_ENDPOINT}
    s3fs_kwargs['config_kwargs'] = {
        's3': {'addressing_style': 'path'},
        'signature_version': 's3v4',
    }

data_path = f"s3://{BUCKET}/{PREFIX}/spans/date=*/*.parquet"
print(f"\nReading: {data_path}")

ddf = dd.read_parquet(
    data_path,
    columns=['start_time_unix_nano', 'duration_ns', 'service_name', 'status_code'],
    storage_options=s3fs_kwargs,
)

total = len(ddf)
print(f"Total spans: {total:,}")
print(f"Partitions:  {ddf.npartitions}")

# Quick stats
sample = ddf.head(5)
print(f"\nSample:")
print(sample)

# Build heatmap (same as otel-navigator)
print("\nBuilding datashader heatmap (timestamp_s vs duration_ms)...")
viz_df = ddf[['start_time_unix_nano', 'duration_ns']].compute()
viz_df['timestamp_s'] = viz_df['start_time_unix_nano'] / 1_000_000_000
viz_df['duration_ms'] = viz_df['duration_ns'] / 1_000_000

points = hv.Points(viz_df, kdims=['timestamp_s', 'duration_ms'])
heatmap = datashade(points, cmap='fire').opts(
    width=900, height=400,
    title=f'OTEL Span Heatmap ({len(viz_df):,} spans) — same view as Panel-Viz',
    xlabel='Timestamp (unix seconds)',
    ylabel='Duration (ms)',
    tools=['wheel_zoom', 'box_zoom', 'reset'],
)
dynspread(heatmap, threshold=0.5, max_px=5)